# Download Wikipedia from HuggingFace
Downloads English Wikipedia articles and saves as .txt files into `data_wikipedia/`.
Target: ~10GB of text. Run `consolidate_data` -> `tokenizer_pipeline` -> `pre_train` after.

In [ ]:
!pip install -q datasets

In [ ]:
import os
from google.colab import drive
from datasets import load_dataset

drive.mount('/content/drive')

OUT_DIR = "/content/drive/MyDrive/synapse/datasets_pretrain/data_wikipedia"
os.makedirs(OUT_DIR, exist_ok=True)

ARTICLES_PER_FILE = 50_000
SEPARATOR = "\n<|endoftext|>\n"

# Check what we already have (for resume)
existing_files = sorted([f for f in os.listdir(OUT_DIR) if f.startswith("wiki_part_") and f.endswith(".txt")])
if existing_files:
    last_num = int(existing_files[-1].replace("wiki_part_", "").replace(".txt", ""))
    articles_done = (last_num + 1) * ARTICLES_PER_FILE
    print(f"Found {len(existing_files)} existing files (~{articles_done:,} articles)")
    print(f"Resuming from article {articles_done:,}...")
else:
    articles_done = 0

print(f"\nDownloading ALL English Wikipedia (~6.8M articles, ~21GB)")
print(f"Streaming from HuggingFace...")

ds = load_dataset("wikimedia/wikipedia", "20231101.en", split="train", streaming=True)

buf = []
file_idx = len(existing_files)
total_chars = 0
skipped = 0

for i, article in enumerate(ds):
    # Skip already-done articles
    if i < articles_done:
        if i % 1_000_000 == 0 and i > 0:
            print(f"  Skipping... {i:,}/{articles_done:,}")
        continue

    text = article["text"].strip()

    # Skip very short articles (stubs, redirects)
    if len(text) < 500:
        skipped += 1
        continue

    # Add title as a header
    title = article.get("title", "")
    if title:
        text = f"{title}\n\n{text}"

    buf.append(text)

    if len(buf) >= ARTICLES_PER_FILE:
        path = os.path.join(OUT_DIR, f"wiki_part_{file_idx:03d}.txt")
        content = SEPARATOR.join(buf)
        with open(path, "w", encoding="utf-8") as f:
            f.write(content)
        size_mb = os.path.getsize(path) / 1024 / 1024
        total_chars += len(content)
        total_gb = total_chars / 1024 / 1024 / 1024
        print(f"  {path.split('/')[-1]}: {size_mb:.0f} MB | {i+1:,} articles | {total_gb:.1f} GB total | {skipped:,} skipped")
        buf = []
        file_idx += 1

# Write remainder
if buf:
    path = os.path.join(OUT_DIR, f"wiki_part_{file_idx:03d}.txt")
    content = SEPARATOR.join(buf)
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    total_chars += len(content)
    file_idx += 1

total_gb = total_chars / 1024 / 1024 / 1024
print(f"\nDone!")
print(f"  {i+1:,} articles processed, {skipped:,} skipped")
print(f"  {file_idx} files, ~{total_gb:.1f} GB")
print(f"  Saved to: {OUT_DIR}")